# Fase ofensiva — Jardine / América

Construcción, progresión y generación de ocasiones. Responde las preguntas 1 y 2 del Plan Maestro (¿cómo juega? ¿cuáles son las ideas del entrenador?) para el lado ofensivo.

In [ ]:
import sys
sys.path.append("../src")
import pandas as pd
import matplotlib.pyplot as plt
from framework_analitico import (
    build_possessions,
    extract_offensive_sequences,
    normalizar_direccion_ataque, 
    identificar_pases_progresivos,
    identificar_conducciones_progresivas,
    zona_inicio_secuencias,
    salida_corta_vs_larga,
    generacion_ocasiones,
)

EQUIPO = "América"  # cambia esto para reaplicar el analisis a otro entrenador/equipo

events_jardine = pd.read_csv("../data/raw/events_jardine_america.csv", low_memory=False)
print(f"Eventos cargados: {len(events_jardine)}")

# Normalizar dirección de ataque ANTES de cualquier análisis de zona/progresión
events_norm = normalizar_direccion_ataque(events_jardine, equipo=EQUIPO)


## 1. Construcción: ¿desde dónde arma el equipo su juego?

In [ ]:
zonas = zona_inicio_secuencias(events_norm, equipo=EQUIPO) # de todas las secuencias ofensivas de América (posesiones con ≥3 pases), ¿en qué tercio del campo arrancan?
print(zonas.zona_inicio.value_counts(normalize=True).mul(100).round(1))

zonas.zona_inicio.value_counts().reindex(["defensivo", "medio", "ofensivo"]).plot(
    kind="bar", color=["#c0392b", "#e67e22", "#27ae60"], title=f"Zona de inicio de secuencias ofensivas — {EQUIPO}"
)
plt.ylabel("Número de secuencias")
plt.tight_layout()
plt.savefig("../outputs/figures/zona_inicio_secuencias.png", dpi=150)
plt.show()


## 2. Construcción: salida corta vs. larga desde el tercio propio

In [ ]:
salida = salida_corta_vs_larga(events_norm, equipo=EQUIPO)
print(salida.tipo_salida.value_counts(normalize=True).mul(100).round(1))
print(f"Longitud promedio de salida: {salida.longitud_m.mean():.1f} m")


## 3. Progresión: pases y conducciones progresivas (criterio StatsBomb: 25% de reducción de distancia a portería)

In [ ]:
pases_prog = identificar_pases_progresivos(events_norm, equipo=EQUIPO)
conducciones_prog = identificar_conducciones_progresivas(events_norm, equipo=EQUIPO)

pct_pases_prog = pases_prog.es_progresivo.mean() * 100
pct_conducciones_prog = conducciones_prog.es_progresiva.mean() * 100

print(f"% de pases que son progresivos: {pct_pases_prog:.1f}%")
print(f"% de conducciones que son progresivas: {pct_conducciones_prog:.1f}%")
print(f"Total pases progresivos: {pases_prog.es_progresivo.sum()}")
print(f"Total conducciones progresivas: {conducciones_prog.es_progresiva.sum()}")

# Por partido, para ver consistencia (relevante para el bloque 5 del Plan Maestro)
prog_por_partido = pases_prog.groupby("match_id").es_progresivo.mean() * 100
print(f"\nDesviación estándar entre partidos: {prog_por_partido.std():.1f} puntos porcentuales")


## 4. Generación de ocasiones: xG y tipo de jugada previa al tiro

In [ ]:
ocasiones = generacion_ocasiones(events_norm, equipo=EQUIPO)
print(f"Total de tiros: {len(ocasiones)}")

if "shot_statsbomb_xg" in ocasiones.columns:
    print(f"xG total: {ocasiones.shot_statsbomb_xg.sum():.2f}")
    print(f"xG promedio por tiro: {ocasiones.shot_statsbomb_xg.mean():.3f}")

    xg_por_patron = ocasiones.groupby("play_pattern").shot_statsbomb_xg.agg(["sum", "count", "mean"]).sort_values("sum", ascending=False)
    print("\nxG por tipo de jugada previa:")
    print(xg_por_patron)

    xg_por_patron["sum"].plot(kind="barh", title=f"xG total por tipo de jugada — {EQUIPO}")
    plt.xlabel("xG acumulado")
    plt.tight_layout()
    plt.savefig("../outputs/figures/xg_por_tipo_jugada.png", dpi=150)
    plt.show()
else:
    print(ocasiones.play_pattern.value_counts())


## 5. Mapas de calor de pases progresivos — origen vs. destino


In [ ]:
# pip install mplsoccer si no lo tienes instalado
from mplsoccer import Pitch
import matplotlib.pyplot as plt

prog = pases_prog[pases_prog.es_progresivo]

pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="black")

fig, axs = plt.subplots(1, 2, figsize=(16, 7))
for ax in axs:
    pitch.draw(ax=ax)

bin_origen = pitch.bin_statistic(prog.x, prog.y, statistic="count", bins=(6, 5))
bin_destino = pitch.bin_statistic(prog.end_x, prog.end_y, statistic="count", bins=(6, 5))

vmax = max(bin_origen["statistic"].max(), bin_destino["statistic"].max())

pitch.heatmap(bin_origen, ax=axs[0], cmap="Greens", vmin=0, vmax=vmax, edgecolors="white")
pitch.heatmap(bin_destino, ax=axs[1], cmap="Greens", vmin=0, vmax=vmax, edgecolors="white")

axs[0].set_title(f"Origen (n={len(prog)})", fontsize=13)
axs[1].set_title("Destino", fontsize=13)
fig.suptitle(f"Pases progresivos — {EQUIPO}: ¿desde dónde y hacia dónde?", fontsize=15)

plt.tight_layout()
fig.savefig("../outputs/figures/heatmap_pases_progresivos.png", dpi=150, bbox_inches="tight")
plt.show()
